# Professional IR Pipeline (PT-BR)

Notebook para iniciar uma implementacao profissional de Information Retrieval para o documento Sindilojas 2025-2026.

Objetivos:
- montar pipeline hibrido (Dense + BM25 + Rerank)
- avaliar com MRR@k e Recall@k
- gerar base para evolucao para RAG em producao

## 1) Set Up Project Environment

In [1]:
%pip install -qU langchain langchain-community langchain-huggingface chromadb sentence-transformers rank-bm25 rapidfuzz pandas numpy tqdm

import os
import re
import json
import hashlib
import random
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import platform
print('Python:', platform.python_version())
print('CWD:', os.getcwd())

SEED = 42
random.seed(SEED)
np.random.seed(SEED)



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Python: 3.12.7
CWD: d:\UC15\senac_ia_uc15_nlp_project\notebooks\pablo


c:\Users\Técnico em IA\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Define Configuration and Constants

In [2]:
@dataclass
class PipelineConfig:
    source_txt_path: str = 'data/pablo/sindilojas_2025_2026.txt'
    gold_files: List[str] = field(default_factory=lambda: [
        'data/pablo/gold_standard.md',
        'data/pablo/gold_standard_deprecated.md',
        'data/core/gold_standard.md',
    ])
    persist_dir: str = 'notebooks/pablo/chroma_professional_db'

    embedding_model_name: str = 'BAAI/bge-m3'
    reranker_model_name: str = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'

    chunk_size_tokens: int = 220
    chunk_overlap_tokens: int = 40

    initial_k_dense: int = 20
    initial_k_bm25: int = 20
    final_k: int = 5

    mrr_k: int = 5
    recall_k: int = 5

    token_overlap_threshold: float = 0.30
    fuzzy_threshold: float = 80.0

    evaluate: bool = False


cfg = PipelineConfig()
cfg

PipelineConfig(source_txt_path='data/pablo/sindilojas_2025_2026.txt', gold_files=['data/pablo/gold_standard.md', 'data/pablo/gold_standard_deprecated.md', 'data/core/gold_standard.md'], persist_dir='notebooks/pablo/chroma_professional_db', embedding_model_name='BAAI/bge-m3', reranker_model_name='cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', chunk_size_tokens=220, chunk_overlap_tokens=40, initial_k_dense=20, initial_k_bm25=20, final_k=5, mrr_k=5, recall_k=5, token_overlap_threshold=0.3, fuzzy_threshold=80.0, evaluate=False)

## 3) Implement Core Data Model

In [3]:
@dataclass
class ChunkRecord:
    chunk_id: str
    content: str
    metadata: Dict[str, Any]


@dataclass
class RetrievedRecord:
    chunk_id: str
    content: str
    metadata: Dict[str, Any]
    dense_rank: Optional[int] = None
    bm25_rank: Optional[int] = None
    rrf_score: Optional[float] = None
    rerank_score: Optional[float] = None

## 4) Implement Main Business Logic

In [4]:
from rapidfuzz import fuzz
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer
from sentence_transformers import CrossEncoder

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma


def normalize_text(text: str) -> str:
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_source_text(raw_text: str) -> str:
    patterns = [
        r'Clicksign\s+\S+',           # qualquer token apos Clicksign
        r'\b\d+\s+de\s+33\b',
        r'SINDICATO DOS COMERCI\w*',   # cobre COMERCIARIOS e variantes
        r'CEP\s*\d{5}-?\d{3}.*?(?:SP|BR)',
    ]

    text = raw_text.replace('\x0c', '\n')
    for p in patterns:
        text = re.sub(p, ' ', text, flags=re.IGNORECASE)

    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()


def parse_gold_markdown(path: Path) -> List[Dict[str, str]]:
    if not path.exists():
        return []

    content = path.read_text(encoding='utf-8', errors='ignore')
    if not content.strip():
        return []

    pattern = re.compile(
        r'Quest[aã]o\s*\d+\s*:\s*(?P<topic>.*?)\n'
        r'Pergunta\s*:\s*(?P<question>.*?)\n'
        r'Resposta\s*:\s*(?P<answer>.*?)(?=\n\s*Quest[aã]o\s*\d+\s*:|\Z)',
        re.IGNORECASE | re.DOTALL,
    )

    items = []
    for m in pattern.finditer(content):
        items.append({
            'topic': m.group('topic').strip(),
            'question': m.group('question').strip(),
            'answer': m.group('answer').strip(),
            'source_file': str(path),
        })
    return items


def detect_clause(text: str) -> Optional[str]:
    m = re.search(r'\b(\d{1,2})\s*[-–]\s*[A-Z][^\n]{3,}', text)
    return m.group(1) if m else None


def make_chunk_id(source: str, idx: int, text: str) -> str:
    payload = f'{source}|{idx}|{text}'
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


In [5]:
class HybridRetrieveRerank:
    def __init__(self, cfg: PipelineConfig):
        self.cfg = cfg
        self._project_root = Path(os.getcwd()).parents[1]

        self.embedding_model = HuggingFaceEmbeddings(
            model_name=cfg.embedding_model_name,
            encode_kwargs={'normalize_embeddings': True},
        )
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.embedding_model_name)
        self.reranker = CrossEncoder(cfg.reranker_model_name)

        self.vectorstore: Optional[Chroma] = None
        self.bm25: Optional[BM25Okapi] = None
        self.chunk_records: List[ChunkRecord] = []
        self.tokenized_chunks: List[List[str]] = []

    def load_and_chunk(self) -> List[ChunkRecord]:
        source_path = self._project_root / self.cfg.source_txt_path
        raw = source_path.read_text(encoding='utf-8', errors='ignore')
        cleaned = clean_source_text(raw)

        splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            tokenizer=self.tokenizer,
            chunk_size=self.cfg.chunk_size_tokens,
            chunk_overlap=self.cfg.chunk_overlap_tokens,
            separators=['\n\n', '\n', '. ', '; ', ', ', ' '],
        )

        splits = splitter.split_text(cleaned)
        records: List[ChunkRecord] = []

        for i, s in enumerate(splits):
            clause = detect_clause(s)
            metadata = {
                'source': str(source_path),
                'chunk_index': i,
                'clause': clause or 'unknown',
            }
            cid = make_chunk_id(str(source_path), i, s)
            records.append(ChunkRecord(chunk_id=cid, content=s, metadata=metadata))

        self.chunk_records = records
        self.tokenized_chunks = [normalize_text(x.content).split() for x in records]
        return records

    def build_indexes(self) -> None:
        docs = [Document(page_content=r.content, metadata={**r.metadata, 'chunk_id': r.chunk_id}) for r in self.chunk_records]
        ids = [r.chunk_id for r in self.chunk_records]
        persist_dir = str(self._project_root / self.cfg.persist_dir)

        self.vectorstore = Chroma.from_documents(
            documents=docs,
            embedding=self.embedding_model,
            ids=ids,
            persist_directory=persist_dir,
        )

        self.bm25 = BM25Okapi(self.tokenized_chunks)

    @staticmethod
    def rrf(rank: int, k: int = 60) -> float:
        return 1.0 / (k + rank)

    def dense_search(self, query: str, k: int) -> List[RetrievedRecord]:
        assert self.vectorstore is not None
        results = self.vectorstore.similarity_search(query, k=k)
        out = []
        for rank, d in enumerate(results, start=1):
            out.append(RetrievedRecord(
                chunk_id=d.metadata.get('chunk_id', ''),
                content=d.page_content,
                metadata=d.metadata,
                dense_rank=rank,
            ))
        return out

    def bm25_search(self, query: str, k: int) -> List[RetrievedRecord]:
        assert self.bm25 is not None
        q_tokens = normalize_text(query).split()
        scores = self.bm25.get_scores(q_tokens)
        top_idx = np.argsort(scores)[::-1][:k]

        out = []
        for rank, idx in enumerate(top_idx, start=1):
            rec = self.chunk_records[int(idx)]
            out.append(RetrievedRecord(
                chunk_id=rec.chunk_id,
                content=rec.content,
                metadata=rec.metadata,
                bm25_rank=rank,
            ))
        return out

    def hybrid_rrf(self, query: str) -> List[RetrievedRecord]:
        dense = self.dense_search(query, self.cfg.initial_k_dense)
        sparse = self.bm25_search(query, self.cfg.initial_k_bm25)

        bucket: Dict[str, RetrievedRecord] = {}

        for r in dense:
            item = bucket.get(r.chunk_id, r)
            item.dense_rank = r.dense_rank
            item.rrf_score = (item.rrf_score or 0.0) + self.rrf(r.dense_rank or 999)
            bucket[r.chunk_id] = item

        for r in sparse:
            item = bucket.get(r.chunk_id, r)
            item.bm25_rank = r.bm25_rank
            item.rrf_score = (item.rrf_score or 0.0) + self.rrf(r.bm25_rank or 999)
            bucket[r.chunk_id] = item

        fused = sorted(bucket.values(), key=lambda x: x.rrf_score or 0.0, reverse=True)
        return fused

    def rerank(self, query: str, candidates: List[RetrievedRecord], top_n: int) -> List[RetrievedRecord]:
        if not candidates:
            return []

        pairs = [[query, c.content] for c in candidates]
        scores = self.reranker.predict(pairs)

        for c, s in zip(candidates, scores):
            c.rerank_score = float(s)

        return sorted(candidates, key=lambda x: x.rerank_score or -1e9, reverse=True)[:top_n]

    def retrieve(self, query: str) -> Dict[str, List[RetrievedRecord]]:
        dense = self.dense_search(query, self.cfg.final_k)
        fused = self.hybrid_rrf(query)
        reranked = self.rerank(query, fused[: max(self.cfg.initial_k_dense, self.cfg.initial_k_bm25)], self.cfg.final_k)
        return {
            'dense_only': dense,
            'hybrid_pre_rerank': fused[: self.cfg.final_k],
            'hybrid_rerank': reranked,
        }


## 5) Add Input/Output Interface

In [6]:
def token_overlap(a: str, b: str) -> float:
    ta = {t for t in normalize_text(a).split() if len(t) > 2}
    tb = {t for t in normalize_text(b).split() if len(t) > 2}
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / max(len(ta), 1)


def is_relevant(expected_answer: str, retrieved_text: str, overlap_thr: float, fuzzy_thr: float) -> bool:
    exp_n = normalize_text(expected_answer)
    ret_n = normalize_text(retrieved_text)

    if not exp_n or not ret_n:
        return False

    if exp_n in ret_n:
        return True

    if token_overlap(exp_n, ret_n) >= overlap_thr:
        return True

    if fuzz.partial_ratio(exp_n, ret_n) >= fuzzy_thr:
        return True

    return False


def mrr_at_k(relevances: List[int], k: int) -> float:
    for i, rel in enumerate(relevances[:k], start=1):
        if rel:
            return 1.0 / i
    return 0.0


def recall_at_k(relevances: List[int], k: int) -> float:
    return float(any(relevances[:k]))


def evaluate_pipeline(pipeline: HybridRetrieveRerank, gold_items: List[Dict[str, str]], cfg: PipelineConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = []

    for item in tqdm(gold_items, desc='Evaluating'):
        q = item['question']
        a = item['answer']
        outputs = pipeline.retrieve(q)

        for stage_name, docs in outputs.items():
            rel = [1 if is_relevant(a, d.content, cfg.token_overlap_threshold, cfg.fuzzy_threshold) else 0 for d in docs]
            rows.append({
                'question': q,
                'stage': stage_name,
                'mrr': mrr_at_k(rel, cfg.mrr_k),
                'recall': recall_at_k(rel, cfg.recall_k),
                'found': int(any(rel)),
                'top_chunk_id': docs[0].chunk_id if docs else None,
                'top_snippet': (docs[0].content[:220] + '...') if docs else '',
            })

    detail_df = pd.DataFrame(rows)
    summary_df = (
        detail_df.groupby('stage', as_index=False)
        .agg(mrr=('mrr', 'mean'), recall=('recall', 'mean'), found_rate=('found', 'mean'))
        .sort_values(['mrr', 'recall'], ascending=False)
    )
    return summary_df, detail_df

In [7]:
# O kernel roda com CWD = notebooks/pablo/, entao subimos 2 niveis para a raiz do projeto
base_path = Path(os.getcwd()).parents[1]

source_path = base_path / cfg.source_txt_path
assert source_path.exists(), f'Arquivo base nao encontrado: {source_path}'

all_gold = []
for rel in cfg.gold_files:
    p = base_path / rel
    parsed = parse_gold_markdown(p)
    if parsed:
        all_gold.extend(parsed)

print(f'Raiz do projeto: {base_path}')
print(f'Gold items carregados: {len(all_gold)}')

pipeline = HybridRetrieveRerank(cfg)
records = pipeline.load_and_chunk()
print(f'Chunks gerados: {len(records)}')

pipeline.build_indexes()
print('Indices dense + BM25 construidos.')


Raiz do projeto: d:\UC15\senac_ia_uc15_nlp_project
Gold items carregados: 20


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12150.57it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunks gerados: 164
Indices dense + BM25 construidos.


## 6) Write Unit Tests for Core Functions

In [8]:
# Testes rapidos das funcoes core
sample_raw = 'SINDICATO DOS COMERCIARIOS\nClicksign abcdef-12345\n  2 de 33\nTexto util.'
cleaned = clean_source_text(sample_raw)
assert 'Clicksign' not in cleaned
assert '2 de 33' not in cleaned
assert 'Texto util' in cleaned

assert normalize_text('Acao') == normalize_text('Ação')
assert token_overlap('salario minimo comerciario', 'valor do salario minimo para comerciario') > 0.3
assert is_relevant('indice de 6,00%', '... aplicacao do indice de 6,00 por cento ...', 0.2, 70.0)
print('OK: testes unitarios basicos passaram.')

OK: testes unitarios basicos passaram.


## 7) Run and Validate the Implementation

In [9]:
summary_df, detail_df = evaluate_pipeline(pipeline, all_gold, cfg)

print('Resumo de metricas:')
display(summary_df)

print('Amostra por pergunta:')
display(detail_df.head(10))

failed = detail_df[detail_df['found'] == 0].copy()
print(f'Perguntas nao encontradas: {len(failed)}')
display(failed[['question', 'stage', 'top_snippet']].head(10))

Evaluating: 100%|██████████| 20/20 [00:22<00:00,  1.14s/it]

Resumo de metricas:


,stage,mrr,recall,found_rate
1,hybrid_pre_rerank,0.8850,0.95,0.95
2,hybrid_rerank,0.8625,1.00,1.00
0,dense_only,0.8500,0.90,0.90


Amostra por pergunta:


,question,stage,mrr,recall,found,top_chunk_id,top_snippet
0,A partir de quando os salários fixos ou parte ...,dense_only,1.0,1.0,1,0cd0c4e09c84d2174bd43f008bd2074d13db5dbdd89228...,1 - REAJUSTAMENTO - Os salários fixos ou parte...
1,A partir de quando os salários fixos ou parte ...,hybrid_pre_rerank,1.0,1.0,1,0cd0c4e09c84d2174bd43f008bd2074d13db5dbdd89228...,1 - REAJUSTAMENTO - Os salários fixos ou parte...
2,A partir de quando os salários fixos ou parte ...,hybrid_rerank,1.0,1.0,1,0cd0c4e09c84d2174bd43f008bd2074d13db5dbdd89228...,1 - REAJUSTAMENTO - Os salários fixos ou parte...
3,Qual o valor mínimo de contribuição deverá ser...,dense_only,1.0,1.0,1,f25ccfafe4dd9bd8c1600b09a53c142ca6f14845bc6684...,"- Filial, estabelecida no município de São Pau..."
4,Qual o valor mínimo de contribuição deverá ser...,hybrid_pre_rerank,1.0,1.0,1,f25ccfafe4dd9bd8c1600b09a53c142ca6f14845bc6684...,"- Filial, estabelecida no município de São Pau..."
5,Qual o valor mínimo de contribuição deverá ser...,hybrid_rerank,1.0,1.0,1,f25ccfafe4dd9bd8c1600b09a53c142ca6f14845bc6684...,"- Filial, estabelecida no município de São Pau..."
6,Quais dias as férias individuais ou coletivas ...,dense_only,1.0,1.0,1,56aa5b15222cb827b554e39e0e0ef4916c5daf7decd413...,Parágrafo 2º - Com a concordância do empregado...
7,Quais dias as férias individuais ou coletivas ...,hybrid_pre_rerank,1.0,1.0,1,56aa5b15222cb827b554e39e0e0ef4916c5daf7decd413...,Parágrafo 2º - Com a concordância do empregado...
8,Quais dias as férias individuais ou coletivas ...,hybrid_rerank,1.0,1.0,1,66dfa142dec1b8f17a0c52495465bdafe79935c5e05e6e...,27 – FÉRIAS - As empresas comunicarão aos seus...
9,A partir de quando os salários fixos ou parte ...,dense_only,1.0,1.0,1,0cd0c4e09c84d2174bd43f008bd2074d13db5dbdd89228...,1 - REAJUSTAMENTO - Os salários fixos ou parte...


Perguntas nao encontradas: 3


,question,stage,top_snippet
45,"Segundo a cláusula do REPIS, qual o salário de...",dense_only,"9 - DO REGIME ESPECIAL DE SALÁRIOS PARA MEIS, ..."
51,Quais são as condições de remuneração e jornad...,dense_only,Parágrafo único - A garantia prevista nesta cl...
52,Quais são as condições de remuneração e jornad...,hybrid_pre_rerank,l) o disposto nesta cláusula não desobriga as ...


In [10]:
demo_questions = [
    'Qual e o indice de reajuste salarial a partir de setembro de 2025?',
    'Qual a garantia minima para comissionista puro?',
    'Qual o valor teto da contribuicao assistencial dos empregados?',
    'Qual o valor da multa por descumprimento de clausula em favor do empregado?',
    'Quais feriados nao autorizam trabalho?',
]

for q in demo_questions:
    print('\n' + '=' * 100)
    print('PERGUNTA:', q)
    out = pipeline.retrieve(q)
    for i, d in enumerate(out['hybrid_rerank'], start=1):
        print(f'[{i}] score={d.rerank_score:.4f} | clause={d.metadata.get("clause")} | id={d.chunk_id[:10]}')
        print(d.content[:300].replace('\n', ' '))
        print('-' * 100)


PERGUNTA: Qual e o indice de reajuste salarial a partir de setembro de 2025?
[1] score=9.5892 | clause=1 | id=0cd0c4e09c
1 - REAJUSTAMENTO - Os salários fixos ou parte fixa dos salários mistos serão reajustados a  partir de 01 de setembro de 2025, a título de recomposição salarial, mediante a aplicação do  índice de 6,00% (seis por cento) incidente sobre os salários já reajustados na Convenção  Coletiva de Trabalho 20
----------------------------------------------------------------------------------------------------
[2] score=2.7715 | clause=2 | id=caa6305481
Parágrafo 5º - Os comerciários com salários superiores ao teto tratado no parágrafo 1º  receberão ABONO PECUNIÁRIO à título de indenização do período de SETEMBRO,  OUTUBRO e NOVEMBRO de 2025, o valor de R$ 2.200,00 (dois mil e duzentos reais),  que poderá ser quitado em 2 (duas) parcelas de igual va
----------------------------------------------------------------------------------------------------
[3] score=2.1509 | clause=unkn

## 8) Calibration to Improve MRR Without Losing Recall

Nesta etapa, combinamos o score do reranker com o score do RRF para evitar que o reranker degrade itens muito relevantes ja bem posicionados no retrieval inicial.

In [11]:
def _minmax(values: List[float]) -> List[float]:
    if not values:
        return []
    vmin, vmax = min(values), max(values)
    if vmax - vmin < 1e-12:
        return [1.0 for _ in values]
    return [(v - vmin) / (vmax - vmin) for v in values]


def retrieve_hybrid_blended(
    pipeline: HybridRetrieveRerank,
    query: str,
    candidate_k: int = 30,
    final_k: int = 5,
    alpha_rerank: float = 0.7,
    beta_rrf: float = 0.3,
    keep_dense_top1: bool = True,
) -> List[RetrievedRecord]:
    fused = pipeline.hybrid_rrf(query)[:candidate_k]
    if not fused:
        return []

    pairs = [[query, c.content] for c in fused]
    rerank_scores = [float(s) for s in pipeline.reranker.predict(pairs)]
    rrf_scores = [float(c.rrf_score or 0.0) for c in fused]

    rerank_n = _minmax(rerank_scores)
    rrf_n = _minmax(rrf_scores)

    for i, c in enumerate(fused):
        c.rerank_score = rerank_scores[i]
        c.blended_score = alpha_rerank * rerank_n[i] + beta_rrf * rrf_n[i]

    ranked = sorted(fused, key=lambda x: x.blended_score, reverse=True)

    if keep_dense_top1:
        dense_top = pipeline.dense_search(query, k=1)
        if dense_top:
            top_id = dense_top[0].chunk_id
            for idx, item in enumerate(ranked):
                if item.chunk_id == top_id and idx > 0:
                    ranked.insert(0, ranked.pop(idx))
                    break

    return ranked[:final_k]


def evaluate_blended(
    pipeline: HybridRetrieveRerank,
    gold_items: List[Dict[str, str]],
    cfg: PipelineConfig,
    candidate_k: int = 30,
    alpha_rerank: float = 0.7,
    beta_rrf: float = 0.3,
    keep_dense_top1: bool = True,
) -> Dict[str, float]:
    mrr_vals = []
    recall_vals = []

    for item in gold_items:
        docs = retrieve_hybrid_blended(
            pipeline=pipeline,
            query=item['question'],
            candidate_k=candidate_k,
            final_k=cfg.final_k,
            alpha_rerank=alpha_rerank,
            beta_rrf=beta_rrf,
            keep_dense_top1=keep_dense_top1,
        )
        rel = [
            1 if is_relevant(item['answer'], d.content, cfg.token_overlap_threshold, cfg.fuzzy_threshold) else 0
            for d in docs
        ]
        mrr_vals.append(mrr_at_k(rel, cfg.mrr_k))
        recall_vals.append(recall_at_k(rel, cfg.recall_k))

    return {
        'stage': 'hybrid_rerank_blended',
        'mrr': float(np.mean(mrr_vals) if mrr_vals else 0.0),
        'recall': float(np.mean(recall_vals) if recall_vals else 0.0),
        'found_rate': float(np.mean([1.0 if x > 0 else 0.0 for x in recall_vals]) if recall_vals else 0.0),
        'candidate_k': candidate_k,
        'alpha_rerank': alpha_rerank,
        'beta_rrf': beta_rrf,
        'keep_dense_top1': keep_dense_top1,
    }


if cfg.evaluate:
    grid_rows = []
    for candidate_k in [20, 30, 40]:
        for alpha in [0.55, 0.65, 0.75, 0.85]:
            beta = 1.0 - alpha
            for keep_top1 in [True, False]:
                metrics = evaluate_blended(
                    pipeline=pipeline,
                    gold_items=all_gold,
                    cfg=cfg,
                    candidate_k=candidate_k,
                    alpha_rerank=alpha,
                    beta_rrf=beta,
                    keep_dense_top1=keep_top1,
                )
                grid_rows.append(metrics)

    blended_grid_df = pd.DataFrame(grid_rows).sort_values(['mrr', 'recall'], ascending=False)
    display(blended_grid_df.head(10))

    best = blended_grid_df.iloc[0].to_dict()
    print('Melhor configuracao blended:')
    print(best)

In [12]:
# Comparacao final entre os estagios antigos e o melhor blended
if cfg.evaluate:
    baseline = summary_df.copy()

    blended_row = pd.DataFrame([
        {
            'stage': 'hybrid_rerank_blended',
            'mrr': best['mrr'],
            'recall': best['recall'],
            'found_rate': best['found_rate'],
        }
    ])

    comparison_df = pd.concat([baseline, blended_row], ignore_index=True)
    comparison_df = comparison_df.sort_values(['mrr', 'recall'], ascending=False)
    display(comparison_df)

    print('Use esta configuracao na inferencia:')
    print(f"candidate_k={int(best['candidate_k'])}, alpha_rerank={best['alpha_rerank']:.2f}, beta_rrf={best['beta_rrf']:.2f}, keep_dense_top1={bool(best['keep_dense_top1'])}")

## 9) Production Inference Defaults

Configuracao consolidada a partir da calibracao:
- candidate_k = 20
- alpha_rerank = 0.55
- beta_rrf = 0.45
- keep_dense_top1 = False

Use esta secao como padrao para inferencia e futuras integracoes com API ou app.

In [13]:
BEST_BLEND_CONFIG = {
    'candidate_k': 20,
    'alpha_rerank': 0.55,
    'beta_rrf': 0.45,
    'keep_dense_top1': False,
}


def retrieve_production(query: str, final_k: Optional[int] = None) -> List[RetrievedRecord]:
    return retrieve_hybrid_blended(
        pipeline=pipeline,
        query=query,
        candidate_k=BEST_BLEND_CONFIG['candidate_k'],
        final_k=final_k or cfg.final_k,
        alpha_rerank=BEST_BLEND_CONFIG['alpha_rerank'],
        beta_rrf=BEST_BLEND_CONFIG['beta_rrf'],
        keep_dense_top1=BEST_BLEND_CONFIG['keep_dense_top1'],
    )


def highlight_snippet(snippet: str, max_len: int = 350) -> str:
    """Formata snippet com quebras de linha preservadas e trunca de forma inteligente."""
    if len(snippet) > max_len:
        snippet = snippet[:max_len].rsplit(' ', 1)[0] + '...'
    
    cleaned = snippet.replace('\n', ' ').strip()
    cleaned = re.sub(r' {2,}', ' ', cleaned)
    return cleaned


def format_results(records: List[RetrievedRecord], max_snippet_len: int = 350) -> pd.DataFrame:
    """Formata resultados com snippets mais legíveis e destacados."""
    rows = []
    for idx, rec in enumerate(records, start=1):
        snippet = highlight_snippet(rec.content, max_snippet_len)
        rows.append({
            'rank': idx,
            'rerank_score': round(float(rec.rerank_score or 0.0), 4),
            'blended_score': round(float(getattr(rec, 'blended_score', 0.0)), 4),
            'clause': rec.metadata.get('clause'),
            'chunk_id': rec.chunk_id[:12],
            'snippet': snippet,
        })
    
    df = pd.DataFrame(rows)
    return df.rename(columns={
        'rerank_score': 'score',
        'blended_score': 'blend',
        'clause': 'cla',
        'chunk_id': 'id',
    })


def display_retrieval_results(query: str, answer_expected: Optional[str] = None):
    """Exibe resultados de forma mais legível com análise de relevância."""
    results = retrieve_production(query)
    
    print(f'\n{"=" * 120}')
    print(f'PERGUNTA: {query}')
    print(f'{"=" * 120}')
    
    for idx, rec in enumerate(results, start=1):
        is_rel = 'YES' if (answer_expected and is_relevant(answer_expected, rec.content, cfg.token_overlap_threshold, cfg.fuzzy_threshold)) else 'NO'
        print(f'\n[{idx}] Relevance={is_rel} | Score={rec.rerank_score:.4f} | Blend={getattr(rec, "blended_score", 0.0):.4f} | Clause={rec.metadata.get("clause")}')
        print(f'    ID: {rec.chunk_id[:16]}')
        print(f'    Snippet: {highlight_snippet(rec.content, 400)}')
        print('    ' + '-' * 116)


print('Configuracao de inferencia carregada:')
print(BEST_BLEND_CONFIG)

Configuracao de inferencia carregada:
{'candidate_k': 20, 'alpha_rerank': 0.55, 'beta_rrf': 0.45, 'keep_dense_top1': False}


In [14]:
production_demo_questions = [
    'Qual e o indice de reajuste salarial estabelecido para setembro de 2025?',
    'Qual a remuneracao minima garantida aos comissionistas puros?',
    'Qual o valor teto da contribuicao assistencial dos empregados?',
]

print('VISUALIZACAO MELHORADA - Opcao 1: Tabela compacta')
for query in production_demo_questions:
    print(f'\n{query}')
    display(format_results(retrieve_production(query)))

print('\n' + '=' * 120)
print('VISUALIZACAO MELHORADA - Opcao 2: Detalhada com contexto')
for query in production_demo_questions[:1]:  # Apenas a primeira para nao poluir
    display_retrieval_results(query)

VISUALIZACAO MELHORADA - Opcao 1: Tabela compacta

Qual e o indice de reajuste salarial estabelecido para setembro de 2025?


,rank,score,blend,cla,id,snippet
0,1,7.8739,0.9618,1,0cd0c4e09c84,1 - REAJUSTAMENTO - Os salários fixos ou parte...
1,2,2.1508,0.8155,unknown,9265141a60ec,Parágrafo 4º - A empresa que não desejar fazer...
2,3,2.6167,0.7827,2,caa630548196,Parágrafo 5º - Os comerciários com salários su...
3,4,0.2459,0.7546,10,5f5cc2006e33,Parágrafo 8º - Observadas as alternativas prev...
4,5,-0.0676,0.6558,unknown,5063bc59d422,DE SÃO PAULO Parágrafo 7º - Antecipações salar...



Qual a remuneracao minima garantida aos comissionistas puros?


,rank,score,blend,cla,id,snippet
0,1,5.1284,1.0000,5,3c74a53c7b1b,DE SÃO PAULO vinte reais); Parágrafo 1º - Empr...
1,2,0.1380,0.7576,4,e55bd940fb83,DE SÃO PAULO 200%; 3 - pagamento em dobro das ...
2,3,-1.4996,0.6929,unknown,733d8c0366e2,"b) manifestação de vontade por escrito, por pa..."
3,4,-2.3524,0.6679,unknown,621e24f16335,DE SÃO PAULO Parágrafo 1º - Os comerciários co...
4,5,-4.9071,0.5168,10,101a70eb8bcd,10 - APRENDIZES - Os empregados que tenham com...



Qual o valor teto da contribuicao assistencial dos empregados?


,rank,score,blend,cla,id,snippet
0,1,3.6572,1.0000,7,b526144ff4e5,7 – DA CONTRIBUIÇÃO ASSISTENCIAL DOS EMPREGADO...
1,2,-2.9270,0.6436,unknown,1c5081ae98e0,DE SÃO PAULO quer seja loja física ou comércio...
2,3,-1.7081,0.6432,unknown,587e97a2d09d,Parágrafo 5º - O recolhimento da contribuição ...
3,4,-2.9111,0.6225,2,caa630548196,Parágrafo 5º - Os comerciários com salários su...
4,5,-4.0748,0.4897,unknown,1a343f8b9050,DE SÃO PAULO trabalhador para acesso aos servi...



VISUALIZACAO MELHORADA - Opcao 2: Detalhada com contexto

PERGUNTA: Qual e o indice de reajuste salarial estabelecido para setembro de 2025?

[1] Relevance=NO | Score=7.8739 | Blend=0.9618 | Clause=1
    ID: 0cd0c4e09c84d217
    Snippet: 1 - REAJUSTAMENTO - Os salários fixos ou parte fixa dos salários mistos serão reajustados a partir de 01 de setembro de 2025, a título de recomposição salarial, mediante a aplicação do índice de 6,00% (seis por cento) incidente sobre os salários já reajustados na Convenção Coletiva de Trabalho 2024/2025, observada a cláusula “EMPREGADOS ADMITIDOS DE 01/09/2024 ATÉ 31/08/2025”. de São Paulo...
    --------------------------------------------------------------------------------------------------------------------

[2] Relevance=NO | Score=2.1508 | Blend=0.8155 | Clause=unknown
    ID: 9265141a60ecc60b
    Snippet: Parágrafo 4º - A empresa que não desejar fazer parcelamento das diferenças salariais tratado no parágrafo 3º acima ou o seu pagamento imediat

## 10) Advanced Snippet Visualization (Optional)

Se preferir uma visualizacao com mais controle sobre formatacao, use as opcoes abaixo.

In [15]:
from IPython.display import HTML, display as ipython_display


def render_snippet_html(rec: RetrievedRecord, query: str, idx: int, max_chars: int = 400) -> HTML:
    """Renderiza um snippet com formatacao HTML para melhor legibilidade."""
    content = rec.content[:max_chars]
    if len(rec.content) > max_chars:
        content = content.rsplit(' ', 1)[0] + '...'
    
    content = content.replace('<', '&lt;').replace('>', '&gt;')
    content = content.replace('\n', '<br/>')
    
    query_words = normalize_text(query).split()
    for word in query_words:
        if len(word) > 3:
            pattern = re.compile(f'\\b{re.escape(word)}\\b', re.IGNORECASE)
            content = pattern.sub(f'<mark style="background-color: #FFFF00;">{word}</mark>', content)
    
    rerank_score = rec.rerank_score or 0.0
    blended_score = getattr(rec, 'blended_score', 0.0)
    
    html_str = f"""
    <div style="border: 1px solid #ddd; margin: 10px 0; padding: 10px; border-radius: 5px; background-color: #f9f9f9;">
        <div style="font-weight: bold; margin-bottom: 8px;">
            [{idx}] Cláusula {rec.metadata.get('clause')} | 
            Rerank: {rerank_score:.4f} | 
            Blended: {blended_score:.4f} | 
            ID: <code style="background-color: #e6f3ff; padding: 2px 5px;">{rec.chunk_id[:16]}</code>
        </div>
        <div style="line-height: 1.6; color: #333; font-size: 14px;">
            {content}
        </div>
    </div>
    """
    return HTML(html_str)


def explore_retrieval(query: str, show_count: int = 5):
    """Interface interativa para explorar e visualizar resultados."""
    results = retrieve_production(query)
    
    ipython_display(HTML(f'<h3 style="color: #1a1a1a; border-bottom: 2px solid #007bff; padding-bottom: 10px;">Pergunta: {query}</h3>'))
    
    for idx, rec in enumerate(results[:show_count], start=1):
        ipython_display(render_snippet_html(rec, query, idx))


# Exemplo de uso:
print('VISUALIZACAO AVANCADA - Com destaque de palavras-chave\n')
explore_retrieval(' Qual o artigo da CLT que autoriza a prática da jornada 12x36?', show_count=3)

VISUALIZACAO AVANCADA - Com destaque de palavras-chave

